# 02 · Representasi Teks: Bag of Words dan TF-IDF
**Mini Project NLP Part 1 · Kelompok 2**
Studi kasus: *Klasifikasi Masukan Pengguna terhadap Pengembangan Aplikasi Digital*

Notebook ini mengubah kolom `clean_text` hasil notebook 01 menjadi vektor angka yang dapat dibaca model machine learning. Tiga representasi dibandingkan:

| Kode | Representasi | Unit | Gagasan singkat |
|---|---|---|---|
| `bow` | Bag of Words | kata (unigram dan bigram) | Menghitung berapa kali setiap kata muncul |
| `tfidf_word` | TF-IDF kata | kata (unigram dan bigram) | Bobot kata dikurangi jika kata itu umum di banyak ulasan |
| `tfidf_char` | TF-IDF potongan huruf | 3 sampai 5 huruf di dalam batas kata (`char_wb`) | Kata yang salah ketik tetap berbagi banyak potongan huruf dengan bentuk benarnya |

| Langkah | Isi |
|---|---|
| 1 | Konfigurasi (mode run otomatis, path, label, parameter vectorizer) |
| 2 | Import library |
| 3 | Memuat `clean_dataset_part1.csv` dan pemeriksaan awal |
| 4 | Split data latih dan uji secara stratified (dipakai juga oleh notebook 03) |
| 5 | Bag of Words: jumlah fitur, contoh matriks sparse |
| 6 | TF-IDF kata: rumus, nilai IDF, contoh matriks sparse |
| 7 | TF-IDF potongan huruf (char n-gram) dan ketahanan terhadap typo |
| 8 | Perbandingan ketiga representasi: jumlah fitur, sparsity, cakupan data uji |
| 9 | Term terpenting per label menurut TF-IDF |
| 10 | Visualisasi dua dimensi TF-IDF |
| 11 | Kelebihan dan kekurangan setiap representasi |
| 12 | Menyimpan artefak untuk notebook 03 |

**Lima label:** Permintaan Fitur, Bug Aplikasi, Desain Antarmuka, Performa Aplikasi, Keamanan Aplikasi.

Prinsip penting: vectorizer **hanya di-fit pada data latih**, lalu dipakai untuk mentransformasi data uji. Dengan begitu kosakata dan nilai IDF tidak "mengintip" data uji.

## 1. Konfigurasi

Notebook ini **tidak perlu diubah kodenya** antara *Run All* dan *Save Version*. Mode run dibaca otomatis dari variabel lingkungan Kaggle `KAGGLE_KERNEL_RUN_TYPE`:

| Nilai | Kapan | Perilaku |
|---|---|---|
| `Interactive` | Run All di editor notebook (juga saat dijalankan lokal) | **Smoke test**: split tetap dihitung pada seluruh data, tetapi vectorizer hanya di-fit pada sampel berstrata `SMOKE_TRAIN_SIZE` ulasan latih dan `SMOKE_TEST_SIZE` ulasan uji |
| `Batch` | Save Version (Save & Run All) | **Full run**: seluruh data latih dan uji dipakai, hasilnya menjadi output resmi |

Split selalu dihitung pada seluruh data, sehingga file split di Output **sama persis** pada kedua mode. Yang diperkecil saat smoke test hanya data untuk fitting vectorizer.

Input dicari otomatis: file `clean_dataset_part1.csv` dicari secara rekursif di bawah `/kaggle/input`. Tambahkan dataset `app-feedback-ewallet-clean` (Output notebook 01) lewat **Add Input** sebelum menjalankan notebook.

Parameter vectorizer ditulis di `REPRESENTATIONS` agar satu sumber kebenaran: dipakai di notebook ini, disimpan ke `representation_config.json`, lalu dibaca notebook 03. Arti setiap parameter dijelaskan di bagian 5 dan 7.

In [ ]:
import os, glob

# ---- mode run otomatis: 'Interactive' (Run All di editor) | 'Batch' (Save Version) ----
RUN_TYPE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive')
SMOKE_TEST = RUN_TYPE != 'Batch'

INPUT_DIR = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working'
FIG_DIR = f'{OUTPUT_DIR}/figures'
CLEAN_FILE = 'clean_dataset_part1.csv'
SPLIT_FILE = 'dataset_split.csv'
CONFIG_FILE = 'representation_config.json'

RANDOM_STATE = 42
TEST_SIZE = 0.2
TEXT_COLUMN = 'clean_text'

SMOKE_TRAIN_SIZE = 3000 if SMOKE_TEST else None   # None = seluruh data latih
SMOKE_TEST_SIZE = 750 if SMOKE_TEST else None     # None = seluruh data uji
SVD_SAMPLE_SIZE = 1500 if SMOKE_TEST else 6000    # titik yang digambar di visualisasi 2D

N_SPARSE_DOCS = 5       # jumlah ulasan pada contoh matriks sparse
N_SPARSE_TERMS = 10     # jumlah kolom pada contoh matriks sparse
TOP_N_TERMS = 10        # jumlah term teratas per label

LABELS = ['Permintaan Fitur', 'Bug Aplikasi', 'Desain Antarmuka', 'Performa Aplikasi', 'Keamanan Aplikasi']
LABEL_COLORS = {
    'Permintaan Fitur': '#0072B2',
    'Bug Aplikasi': '#D55E00',
    'Desain Antarmuka': '#CC79A7',
    'Performa Aplikasi': '#E69F00',
    'Keamanan Aplikasi': '#009E73',
}

# Parameter vectorizer (ngram_range ditulis sebagai list agar dapat disimpan ke JSON)
REPRESENTATIONS = {
    'bow': {
        'name': 'Bag of Words',
        'class': 'CountVectorizer',
        'params': {'analyzer': 'word', 'token_pattern': r'\S+', 'ngram_range': [1, 2],
                   'min_df': 2, 'max_df': 0.95, 'lowercase': False},
    },
    'tfidf_word': {
        'name': 'TF-IDF kata',
        'class': 'TfidfVectorizer',
        'params': {'analyzer': 'word', 'token_pattern': r'\S+', 'ngram_range': [1, 2],
                   'min_df': 2, 'max_df': 0.95, 'lowercase': False,
                   'sublinear_tf': True, 'smooth_idf': True, 'norm': 'l2'},
    },
    'tfidf_char': {
        'name': 'TF-IDF char n-gram',
        'class': 'TfidfVectorizer',
        'params': {'analyzer': 'char_wb', 'ngram_range': [3, 5],
                   'min_df': 5, 'max_df': 0.95, 'lowercase': False,
                   'sublinear_tf': True, 'smooth_idf': True, 'norm': 'l2'},
    },
}

print(f'RUN_TYPE : {RUN_TYPE} -> {"SMOKE TEST" if SMOKE_TEST else "FULL RUN"}')
print(f'Fitting  : {"sampel " + str(SMOKE_TRAIN_SIZE) + " latih / " + str(SMOKE_TEST_SIZE) + " uji" if SMOKE_TEST else "seluruh data latih dan uji"}')

## 2. Import library

| Library | Kegunaan |
|---|---|
| `pandas`, `numpy` | Mengolah tabel data |
| `scipy.sparse` | Format matriks sparse (hanya nilai bukan nol yang disimpan) |
| `scikit-learn` | `train_test_split`, `CountVectorizer`, `TfidfVectorizer`, `chi2`, `TruncatedSVD` |
| `matplotlib` | Grafik |
| `json`, `time` | Menyimpan konfigurasi dan mengukur waktu fitting |

Semua library sudah tersedia di Kaggle, sehingga notebook ini tidak membutuhkan Internet.

In [ ]:
import json, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
os.makedirs(FIG_DIR, exist_ok=True)

VECTORIZER_CLASSES = {'CountVectorizer': CountVectorizer, 'TfidfVectorizer': TfidfVectorizer}


def build_vectorizer(key: str):
    """Buat vectorizer baru dari REPRESENTATIONS[key] (ngram_range list diubah menjadi tuple)."""
    spec = REPRESENTATIONS[key]
    params = dict(spec['params'], ngram_range=tuple(spec['params']['ngram_range']))
    return VECTORIZER_CLASSES[spec['class']](**params)


def save_fig(fig: plt.Figure, name: str) -> None:
    """Simpan grafik ke OUTPUT_DIR/figures dengan resolusi 200 dpi untuk laporan dan slide."""
    fig.savefig(f'{FIG_DIR}/{name}.png', dpi=200, bbox_inches='tight')

## 3. Memuat dataset

`clean_dataset_part1.csv` adalah Output notebook 01 dengan kolom `text` (ulasan asli), `clean_text` (hasil preprocessing), `label`, `app`, `rating`, dan `source`.

File dibaca dengan `keep_default_na=False` agar kata seperti `null` atau `nan` di dalam ulasan tidak berubah menjadi nilai kosong. Setiap baris diberi `row_id`, yaitu nomor urut baris di file asli (mulai 0). `row_id` inilah yang dipakai untuk mencatat split, sehingga notebook lain dapat mencocokkan baris dengan pasti.

In [ ]:
clean_paths = sorted(glob.glob(f'{INPUT_DIR}/**/{CLEAN_FILE}', recursive=True))
if not clean_paths:
    raise FileNotFoundError(f'{CLEAN_FILE} tidak ditemukan di {INPUT_DIR}. Tambahkan dataset '
                            '"app-feedback-ewallet-clean" (output notebook 01) lewat Add Input.')
CLEAN_PATH = clean_paths[0]
print('Membaca:', CLEAN_PATH)

df = pd.read_csv(CLEAN_PATH, keep_default_na=False)
df.insert(0, 'row_id', np.arange(len(df)))
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)

unknown_labels = set(df['label']) - set(LABELS)
n_empty = df[TEXT_COLUMN].str.strip().eq('').sum()
print('Ukuran                  :', df.shape)
print('Label di luar LABELS    :', unknown_labels or 'tidak ada')
print('clean_text kosong       :', n_empty)
assert not unknown_labels, 'Ada label yang tidak dikenal, periksa notebook 01.'
assert n_empty == 0, 'Ada clean_text kosong, periksa notebook 01.'

label_counts = df['label'].value_counts().reindex(LABELS)
display(pd.DataFrame({'jumlah': label_counts, 'persen': (label_counts / len(df) * 100).round(2)}))
df[['row_id', 'text', TEXT_COLUMN, 'label']].head()

## 4. Split data latih dan uji

Data dibagi **80 persen latih dan 20 persen uji** dengan `train_test_split(test_size=0.2, stratify=label, random_state=42)`.

- `stratify=label` membuat proporsi setiap label di data latih dan uji sama dengan data penuh. Ini penting karena label sangat tidak seimbang; tanpa stratifikasi, label kecil seperti Desain Antarmuka bisa terlalu sedikit di data uji.
- `random_state=42` membuat pembagian selalu sama setiap kali notebook dijalankan.
- Split dihitung **satu kali untuk seluruh proyek** dan disimpan sebagai kolom `split` di `dataset_split.csv`. Notebook 03 (dan model Part 2) membaca file ini, sehingga semua model diuji pada ulasan uji yang sama dan hasilnya dapat dibandingkan secara adil.

In [ ]:
train_ids, test_ids = train_test_split(
    df['row_id'].to_numpy(), test_size=TEST_SIZE, stratify=df['label'], random_state=RANDOM_STATE)

df['split'] = 'train'
df.loc[df['row_id'].isin(test_ids), 'split'] = 'test'

split_table = (pd.crosstab(df['label'], df['split'], margins=True, margins_name='Total')
               .reindex(index=LABELS + ['Total'], columns=['train', 'test', 'Total']))
split_table['persen_test'] = (split_table['test'] / split_table['Total'] * 100).round(2)
split_table

### 4.1 Pemeriksaan kebocoran: `clean_text` kembar di latih dan uji

Notebook 01 sudah membuang ulasan hampir kembar berdasarkan teks asli. Namun dua ulasan asli yang berbeda bisa menghasilkan `clean_text` yang sama setelah preprocessing (misalnya `aplikasi lemot` dan `aplikasinya lemot bgt`). Sel ini menghitung berapa ulasan uji yang `clean_text`-nya persis ada di data latih.

Baris ini **tidak dibuang**, karena ulasan pendek yang sama memang wajar terjadi di dunia nyata (banyak orang menulis `aplikasi lemot`). Angkanya dicatat sebagai keterbatasan di laporan: model akan lebih mudah menebak ulasan uji seperti ini.

In [ ]:
train_texts = set(df.loc[df['split'] == 'train', TEXT_COLUMN])
test_df_full = df[df['split'] == 'test']
in_train = test_df_full[TEXT_COLUMN].isin(train_texts)
print(f'Ulasan uji dengan clean_text yang juga ada di data latih: {in_train.sum():,} '
      f'dari {len(test_df_full):,} ({in_train.mean() * 100:.2f} persen)')
test_df_full.loc[in_train, TEXT_COLUMN].value_counts().head(10).rename('jumlah_di_uji').to_frame()

### 4.2 Data untuk fitting vectorizer

Pada Save Version seluruh data latih dan uji dipakai. Pada smoke test diambil sampel berstrata (proporsi label tetap) dari data latih dan uji, dengan `random_state` tetap. Hasil smoke test hanya untuk memastikan semua sel berjalan; angka resmi diambil dari Save Version.

In [ ]:
def stratified_subset(frame: pd.DataFrame, n: int | None) -> pd.DataFrame:
    """Ambil n baris berstrata per label; None berarti seluruh baris."""
    if n is None or n >= len(frame):
        return frame
    subset, _ = train_test_split(frame, train_size=n, stratify=frame['label'], random_state=RANDOM_STATE)
    return subset.sort_values('row_id')

train_df = stratified_subset(df[df['split'] == 'train'], SMOKE_TRAIN_SIZE)
test_df = stratified_subset(df[df['split'] == 'test'], SMOKE_TEST_SIZE)
y_train, y_test = train_df['label'].to_numpy(), test_df['label'].to_numpy()
print(f'Data latih untuk fitting : {len(train_df):,} ulasan')
print(f'Data uji untuk transform : {len(test_df):,} ulasan')

## 5. Bag of Words

Bag of Words (BoW) memandang ulasan sebagai "kantong kata": urutan kata diabaikan, yang dicatat hanya **berapa kali setiap kata muncul**. Setiap ulasan menjadi satu baris vektor dengan panjang sama dengan jumlah kata di kosakata.

Parameter `CountVectorizer` yang dipakai:

| Parameter | Nilai | Alasan |
|---|---|---|
| `analyzer='word'` | kata | Unit fitur adalah kata |
| `token_pattern=r'\S+'` | setiap potongan tanpa spasi | `clean_text` sudah ditokenisasi di notebook 01, jadi cukup dipisah per spasi. Pola bawaan scikit-learn membuang token satu huruf, dan itu tidak diinginkan |
| `ngram_range=(1, 2)` | unigram dan bigram | Bigram menangkap frasa dua kata seperti `tidak bisa` atau `saldo hilang` yang maknanya berbeda dari kata tunggalnya |
| `min_df=2` | minimal muncul di 2 ulasan latih | Kata atau frasa yang hanya muncul sekali biasanya typo atau terlalu khusus, dan tidak membantu model menggeneralisasi |
| `max_df=0.95` | maksimal muncul di 95 persen ulasan | Membuang term yang muncul di hampir semua ulasan karena tidak membedakan label |
| `lowercase=False` | tidak diubah | Case folding sudah dilakukan di notebook 01 |

Vectorizer di-fit (`fit_transform`) pada data latih untuk membangun kosakata, lalu hanya `transform` pada data uji. Kata di data uji yang tidak ada di kosakata latih diabaikan.

In [ ]:
vectorizers, X_train, X_test, fit_seconds = {}, {}, {}, {}

def fit_representation(key: str) -> None:
    """Fit vectorizer pada data latih, transform data uji, dan catat waktunya."""
    start = time.perf_counter()
    vec = build_vectorizer(key)
    X_train[key] = vec.fit_transform(train_df[TEXT_COLUMN])
    X_test[key] = vec.transform(test_df[TEXT_COLUMN])
    fit_seconds[key] = time.perf_counter() - start
    vectorizers[key] = vec
    print(f'{REPRESENTATIONS[key]["name"]:<20}: latih {X_train[key].shape}, uji {X_test[key].shape}, '
          f'{fit_seconds[key]:.1f} detik')

fit_representation('bow')

bow_vocab = vectorizers['bow'].get_feature_names_out()
n_unigram = sum(' ' not in t for t in bow_vocab)
print(f'Jumlah fitur BoW : {len(bow_vocab):,} (unigram {n_unigram:,}, bigram {len(bow_vocab) - n_unigram:,})')

### 5.1 Jumlah fitur menurut pilihan parameter

Tabel berikut menunjukkan pengaruh `ngram_range` dan `min_df` terhadap ukuran kosakata. Tanpa batas `min_df`, kosakata membengkak oleh term yang hanya muncul sekali. Konfigurasi yang dipakai di notebook ini ditandai di kolom `dipakai`.

In [ ]:
vocab_rows = []
for ngram in [(1, 1), (1, 2)]:
    for min_df in [1, 2, 5]:
        vec = CountVectorizer(token_pattern=r'\S+', ngram_range=ngram, min_df=min_df, max_df=0.95, lowercase=False)
        n_features = vec.fit(train_df[TEXT_COLUMN]).get_feature_names_out().size
        vocab_rows.append({'ngram_range': str(ngram), 'min_df': min_df, 'jumlah_fitur': n_features,
                           'dipakai': 'ya' if (ngram == (1, 2) and min_df == 2) else ''})
pd.DataFrame(vocab_rows)

### 5.2 Contoh representasi sparse

Hampir semua nilai di matriks BoW adalah nol, karena satu ulasan hanya memuat belasan kata dari ribuan kata di kosakata. Karena itu scikit-learn menyimpannya sebagai **matriks sparse** (format CSR): hanya posisi dan nilai yang bukan nol yang disimpan.

Dua tampilan diberikan untuk `N_SPARSE_DOCS` ulasan latih:
1. **Format koordinat**: daftar `(baris, kolom, nilai)` untuk setiap nilai bukan nol, persis seperti yang disimpan komputer.
2. **Potongan matriks**: tabel ulasan x term untuk beberapa term yang muncul di ulasan contoh, agar terlihat bentuk vektornya (sebagian besar sel tetap nol).

Ulasan contoh dipilih satu per label (ditambah ulasan berikutnya bila perlu) dengan `random_state` tetap.

In [ ]:
sample_pos = (train_df.reset_index(drop=True).groupby('label', sort=False)
              .sample(1, random_state=RANDOM_STATE).index[:N_SPARSE_DOCS].to_numpy())
sample_docs = train_df.iloc[sample_pos][['row_id', TEXT_COLUMN, 'label']].reset_index(drop=True)
display(sample_docs)


def readable_names(key: str) -> np.ndarray:
    """Nama fitur; pada char n-gram spasi penanda batas kata ditampilkan sebagai '_'."""
    names = vectorizers[key].get_feature_names_out()
    return np.char.replace(names.astype(str), ' ', '_') if key == 'tfidf_char' else names


def sparse_triplets(key: str, n_rows: int = 12) -> pd.DataFrame:
    """Nilai bukan nol untuk ulasan contoh dalam format koordinat (baris, kolom, term, nilai)."""
    coo = X_train[key][sample_pos].tocoo()
    names = readable_names(key)
    table = pd.DataFrame({'baris': coo.row, 'kolom': coo.col, 'term': names[coo.col], 'nilai': coo.data})
    return table.sort_values(['baris', 'kolom']).head(n_rows).round(4).reset_index(drop=True)


def dense_slice(key: str) -> pd.DataFrame:
    """Potongan matriks: ulasan contoh x N_SPARSE_TERMS term yang paling sering di ulasan contoh."""
    sub = X_train[key][sample_pos]
    names = readable_names(key)
    top_cols = np.argsort(-np.asarray((sub > 0).sum(axis=0)).ravel(), kind='stable')[:N_SPARSE_TERMS]
    top_cols = np.sort(top_cols)
    return pd.DataFrame(sub[:, top_cols].toarray(), columns=names[top_cols],
                        index=[f'ulasan_{i}' for i in range(len(sample_pos))]).round(3)


sub_bow = X_train['bow'][sample_pos]
print(f'Ukuran potongan: {sub_bow.shape}, nilai bukan nol: {sub_bow.nnz}, '
      f'nol: {sub_bow.shape[0] * sub_bow.shape[1] - sub_bow.nnz:,}')
display(sparse_triplets('bow'))
dense_slice('bow')

## 6. TF-IDF kata

TF-IDF (*Term Frequency - Inverse Document Frequency*) memberi bobot pada setiap term:

$$\text{tfidf}(t, d) = \text{tf}(t, d) \times \text{idf}(t), \qquad \text{idf}(t) = \ln\frac{1 + n}{1 + \text{df}(t)} + 1$$

- $\text{tf}(t, d)$: berapa kali term $t$ muncul di ulasan $d$. Dengan `sublinear_tf=True` nilainya diganti $1 + \ln(\text{tf})$, sehingga kata yang diulang-ulang dalam satu ulasan (misalnya `lama lama lama`) tidak mendominasi.
- $n$: jumlah ulasan latih, $\text{df}(t)$: jumlah ulasan latih yang memuat $t$. Term yang muncul di banyak ulasan (misalnya `aplikasi`) mendapat IDF kecil, sedangkan term khas (misalnya `fitur gelap`) mendapat IDF besar.
- Rumus IDF di atas adalah bawaan scikit-learn dengan `smooth_idf=True` (penambahan 1 mencegah pembagian dengan nol).
- `norm='l2'`: setiap vektor ulasan dibagi panjangnya sehingga panjang vektornya 1. Ulasan panjang dan pendek menjadi sebanding.

Parameter kosakata (`token_pattern`, `ngram_range=(1, 2)`, `min_df=2`, `max_df=0.95`) sama dengan BoW, sehingga **kosakatanya identik** dan perbedaannya hanya pada bobot.

In [ ]:
fit_representation('tfidf_word')
tfidf_vocab = vectorizers['tfidf_word'].get_feature_names_out()
assert (tfidf_vocab == bow_vocab).all(), 'Kosakata BoW dan TF-IDF kata seharusnya sama.'
print(f'Jumlah fitur TF-IDF kata: {len(tfidf_vocab):,} (sama dengan BoW)')

### 6.1 Nilai IDF: term paling umum dan paling khas

Term dengan IDF terkecil adalah term yang muncul di paling banyak ulasan, sehingga bobotnya diturunkan. Term dengan IDF terbesar hanya muncul di sedikit ulasan (minimal `min_df`). Kolom `df` adalah jumlah ulasan latih yang memuat term tersebut.

In [ ]:
idf = vectorizers['tfidf_word'].idf_
doc_freq = np.asarray((X_train['bow'] > 0).sum(axis=0)).ravel()
idf_table = pd.DataFrame({'term': tfidf_vocab, 'df': doc_freq, 'idf': idf.round(3)})
order = idf_table.sort_values(['idf', 'term']).reset_index(drop=True)
pd.concat({'IDF terkecil (umum)': order.head(10).reset_index(drop=True),
           'IDF terbesar (khas)': order.tail(10).reset_index(drop=True)}, axis=1)

### 6.2 Contoh representasi sparse TF-IDF

Ulasan contoh yang sama seperti 5.2. Posisi nilai bukan nol **sama persis** dengan BoW (kosakata sama), tetapi nilainya kini bobot pecahan antara 0 dan 1. Term umum mendapat bobot lebih kecil daripada term khas dalam ulasan yang sama.

In [ ]:
display(sparse_triplets('tfidf_word'))
dense_slice('tfidf_word')

## 7. TF-IDF potongan huruf (char n-gram)

Notebook 01 sudah mengoreksi banyak typo, tetapi tidak semuanya bisa tertangkap (misalnya typo yang jarang muncul atau bentuk singkatan baru). Pada representasi kata, `verifikasi` dan `verifkasi` adalah dua fitur yang sama sekali berbeda. Jika `verifkasi` hanya muncul di data uji, informasinya hilang.

TF-IDF char n-gram memecah teks menjadi **potongan huruf** sepanjang 3 sampai 5. Dengan `analyzer='char_wb'`, potongan hanya diambil **di dalam batas kata**, dan setiap kata diberi spasi di awal dan akhir. Contoh untuk kata `eror`: ` er`, `ero`, `ror`, `or `, ` ero`, `eror`, `ror `, ` eror`, `eror `. Kata `error` berbagi sebagian besar potongan tersebut, sehingga vektornya mirip.

| Parameter | Nilai | Alasan |
|---|---|---|
| `analyzer='char_wb'` | potongan huruf di dalam kata | Potongan tidak melintasi dua kata, sehingga jumlah fitur lebih terkendali dan lebih bermakna daripada `analyzer='char'` |
| `ngram_range=(3, 5)` | 3 sampai 5 huruf | Potongan 1 sampai 2 huruf terlalu umum. Potongan 3 sampai 5 huruf kira-kira seukuran suku kata atau imbuhan |
| `min_df=5` | minimal di 5 ulasan latih | Jumlah potongan huruf jauh lebih banyak daripada kata, sehingga batas dinaikkan untuk membuang potongan yang sangat jarang |
| `max_df=0.95`, `sublinear_tf=True`, `norm='l2'` | sama dengan TF-IDF kata | Agar perbandingan adil |

In [ ]:
fit_representation('tfidf_char')
char_vocab = vectorizers['tfidf_char'].get_feature_names_out()
print(f'Jumlah fitur TF-IDF char n-gram: {len(char_vocab):,}')

analyzer = vectorizers['tfidf_char'].build_analyzer()
print('Potongan huruf untuk "eror":', analyzer('eror'))

### 7.1 Ketahanan terhadap typo

Untuk setiap pasangan (kata benar, kata salah ketik), dihitung **cosine similarity** antara vektor keduanya. Nilai 1 berarti identik, 0 berarti tidak ada kesamaan sama sekali.

- Pada TF-IDF kata, dua kata yang berbeda ejaan selalu bernilai 0, dan kata salah ketik biasanya tidak ada di kosakata (vektornya kosong).
- Pada TF-IDF char n-gram, nilainya tinggi karena kedua kata berbagi banyak potongan huruf.

Pasangan typo di bawah ini adalah contoh buatan untuk ilustrasi.

In [ ]:
TYPO_PAIRS = [('verifikasi', 'verifkasi'), ('transaksi', 'transaki'), ('saldo', 'sado'),
              ('lambat', 'lambt'), ('aplikasi', 'aplikasih'), ('keamanan', 'kemanan'), ('error', 'eror')]

rows = []
for correct, typo in TYPO_PAIRS:
    row = {'kata_benar': correct, 'kata_typo': typo}
    for key in ['tfidf_word', 'tfidf_char']:
        vec = vectorizers[key]
        pair = vec.transform([correct, typo])
        row[f'{key}_typo_di_kosakata'] = 'kosong' if pair[1].nnz == 0 else 'ada'
        row[f'cosine_{key}'] = round(float(cosine_similarity(pair[0], pair[1])[0, 0]), 3)
    rows.append(row)
typo_table = pd.DataFrame(rows)
typo_table

### 7.2 Contoh representasi sparse char n-gram

Ulasan contoh yang sama seperti 5.2. Jumlah nilai bukan nol per ulasan jauh lebih banyak daripada representasi kata, karena satu kata menghasilkan beberapa potongan huruf. Agar mudah dibaca, spasi penanda awal atau akhir kata ditampilkan sebagai `_` (misalnya `_ero` adalah potongan ` ero` di awal kata `eror`).

In [ ]:
sub_char = X_train['tfidf_char'][sample_pos]
print(f'Nilai bukan nol per ulasan contoh: {np.diff(sub_char.indptr).tolist()}')
display(sparse_triplets('tfidf_char'))
dense_slice('tfidf_char')

## 8. Perbandingan ketiga representasi

Ukuran yang dibandingkan:

| Kolom | Arti |
|---|---|
| `jumlah_fitur` | Panjang vektor (ukuran kosakata) |
| `nnz_latih` | Jumlah nilai bukan nol di matriks latih |
| `sparsity_persen` | Persentase sel bernilai nol: $1 - \frac{\text{nnz}}{\text{baris} \times \text{kolom}}$ |
| `rata_nnz_per_ulasan` | Rata-rata jumlah fitur aktif per ulasan |
| `ukuran_mb` | Memori matriks sparse latih (data, indeks kolom, dan penunjuk baris) |
| `uji_vektor_kosong` | Ulasan uji yang seluruh termnya tidak ada di kosakata latih, sehingga vektornya nol semua dan model tidak punya informasi apa pun |
| `uji_term_di_luar_kosakata_persen` | Persentase kata di ulasan uji yang tidak ada di kosakata latih (untuk char n-gram dihitung per potongan huruf) |
| `waktu_fit_detik` | Waktu fit latih dan transform uji |

In [ ]:
def oov_rate(key: str) -> float:
    """Persentase token (atau potongan huruf) uji yang tidak ada di kosakata latih."""
    vec = vectorizers[key]
    analyze = vec.build_analyzer()
    vocab = vec.vocabulary_
    total = missing = 0
    for text in test_df[TEXT_COLUMN]:
        terms = analyze(text)
        if vec.analyzer == 'word':
            terms = [t for t in terms if ' ' not in t]   # hanya unigram agar mudah ditafsirkan
        total += len(terms)
        missing += sum(t not in vocab for t in terms)
    return missing / total * 100


summary_rows = []
for key, spec in REPRESENTATIONS.items():
    Xtr = X_train[key]
    summary_rows.append({
        'representasi': spec['name'],
        'ngram_range': str(tuple(spec['params']['ngram_range'])),
        'min_df': spec['params']['min_df'],
        'jumlah_fitur': Xtr.shape[1],
        'nnz_latih': Xtr.nnz,
        'sparsity_persen': round((1 - Xtr.nnz / (Xtr.shape[0] * Xtr.shape[1])) * 100, 4),
        'rata_nnz_per_ulasan': round(Xtr.nnz / Xtr.shape[0], 1),
        'ukuran_mb': round((Xtr.data.nbytes + Xtr.indices.nbytes + Xtr.indptr.nbytes) / 1e6, 2),
        'uji_vektor_kosong': int((np.diff(X_test[key].indptr) == 0).sum()),
        'uji_term_di_luar_kosakata_persen': round(oov_rate(key), 2),
        'waktu_fit_detik': round(fit_seconds[key], 1),
    })
feature_summary = pd.DataFrame(summary_rows).set_index('representasi')
feature_summary

### 8.1 Grafik jumlah fitur dan fitur aktif per ulasan

Dua ukuran ini berbeda skala, sehingga digambar di dua panel terpisah. Panel kiri menunjukkan panjang vektor, panel kanan menunjukkan berapa fitur yang rata-rata aktif per ulasan. Keduanya menjelaskan mengapa char n-gram lebih berat dihitung.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
names = feature_summary.index.tolist()
bar_color = '#4C72B0'
for ax, col, title in [(axes[0], 'jumlah_fitur', 'Jumlah fitur (ukuran kosakata)'),
                       (axes[1], 'rata_nnz_per_ulasan', 'Rata-rata fitur aktif per ulasan')]:
    values = feature_summary[col]
    bars = ax.barh(names[::-1], values[::-1], color=bar_color, height=0.55)
    ax.bar_label(bars, labels=[f'{v:,.0f}' if v >= 100 else f'{v:,.1f}' for v in values[::-1]], padding=4)
    ax.set_title(title)
    ax.set_xlabel(col.replace('_', ' '))
    ax.set_xlim(0, values.max() * 1.2)
    ax.grid(axis='x', alpha=0.3)
fig.suptitle('Perbandingan ukuran representasi teks (data latih)', fontsize=13)
plt.tight_layout()
save_fig(fig, '01_feature_counts')
plt.show()

## 9. Term terpenting per label menurut TF-IDF

Dua cara melihat term penting per label, keduanya dihitung pada data latih TF-IDF kata:

1. **Rata-rata bobot TF-IDF** setiap term pada ulasan berlabel tersebut. Cara ini menunjukkan term yang paling sering dan kuat muncul di label itu, tetapi term umum seperti `tidak` atau `aplikasi` bisa ikut muncul di beberapa label.
2. **Uji chi-square (satu label melawan label lain)**. Cara ini mengukur seberapa tidak seimbang kemunculan term di label tersebut dibanding label lain, sehingga yang muncul adalah term pembeda. Hanya term yang rata-ratanya lebih tinggi di label tersebut yang diambil.

Karena label dibuat dari aturan kata kunci di notebook 00, term pembeda yang muncul sebagian besar adalah kata kunci pelabelan itu sendiri. Ini wajar dan perlu disebutkan di laporan sebagai keterbatasan (model ikut mempelajari aturan kata kunci).

In [ ]:
X_tfidf_train = X_train['tfidf_word']
top_mean, top_chi2, top_rows = {}, {}, []
for lab in LABELS:
    in_label = y_train == lab
    mean_in = np.asarray(X_tfidf_train[in_label].mean(axis=0)).ravel()
    mean_out = np.asarray(X_tfidf_train[~in_label].mean(axis=0)).ravel()
    chi_scores, _ = chi2(X_tfidf_train, in_label)
    chi_scores = np.where(mean_in > mean_out, np.nan_to_num(chi_scores), 0.0)

    idx_mean = np.argsort(-mean_in, kind='stable')[:TOP_N_TERMS]
    idx_chi = np.argsort(-chi_scores, kind='stable')[:TOP_N_TERMS]
    top_mean[lab] = list(zip(tfidf_vocab[idx_mean], mean_in[idx_mean]))
    top_chi2[lab] = list(zip(tfidf_vocab[idx_chi], chi_scores[idx_chi]))
    for rank, (i_m, i_c) in enumerate(zip(idx_mean, idx_chi), 1):
        top_rows.append({'label': lab, 'peringkat': rank,
                         'term_rata_tfidf': tfidf_vocab[i_m], 'rata_tfidf': round(mean_in[i_m], 4),
                         'term_chi2': tfidf_vocab[i_c], 'skor_chi2': round(chi_scores[i_c], 2)})
top_terms = pd.DataFrame(top_rows)

print('Term dengan rata-rata TF-IDF tertinggi per label')
display(pd.DataFrame({lab: [t for t, _ in top_mean[lab]] for lab in LABELS}, index=range(1, TOP_N_TERMS + 1)))
print('Term pembeda per label (chi-square)')
display(pd.DataFrame({lab: [t for t, _ in top_chi2[lab]] for lab in LABELS}, index=range(1, TOP_N_TERMS + 1)))

### 9.1 Grafik term pembeda per label

Setiap panel menampilkan term dengan rata-rata bobot TF-IDF tertinggi di label tersebut, diurutkan dari skor chi-square terbesar (term pembeda). Panjang batang adalah rata-rata bobot TF-IDF term itu pada ulasan berlabel tersebut, warna mengikuti `LABEL_COLORS` yang sama dengan notebook 01.

In [ ]:
mean_by_label = {lab: np.asarray(X_tfidf_train[y_train == lab].mean(axis=0)).ravel() for lab in LABELS}
vocab_index = {t: i for i, t in enumerate(tfidf_vocab)}

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for ax, lab in zip(axes.flat, LABELS):
    terms = [t for t, _ in top_chi2[lab]]
    values = [mean_by_label[lab][vocab_index[t]] for t in terms]
    ax.barh(terms[::-1], values[::-1], color=LABEL_COLORS[lab], height=0.6)
    ax.set_title(lab)
    ax.set_xlabel('Rata-rata bobot TF-IDF')
    ax.grid(axis='x', alpha=0.3)
axes.flat[-1].axis('off')
fig.suptitle(f'{TOP_N_TERMS} term pembeda per label (chi-square, TF-IDF kata, data latih)', fontsize=14)
plt.tight_layout()
save_fig(fig, '02_top_terms_per_label')
plt.show()

## 10. Visualisasi dua dimensi TF-IDF

Vektor TF-IDF berdimensi ribuan sehingga tidak bisa digambar langsung. `TruncatedSVD` (setara PCA untuk matriks sparse, juga dikenal sebagai LSA) memadatkan vektor menjadi 2 komponen yang menyimpan variasi terbesar. Hanya `SVD_SAMPLE_SIZE` ulasan latih yang digambar agar grafik tidak terlalu padat, dan label dengan jumlah ulasan lebih sedikit digambar di lapisan atas agar tidak tertutup label mayoritas.

Grafik ini hanya gambaran kasar: 2 komponen menyimpan sebagian kecil informasi (lihat persentase variansi di judul sumbu). Label yang menumpuk di grafik belum tentu sulit dibedakan oleh model yang memakai seluruh dimensi.

In [ ]:
n_points = min(SVD_SAMPLE_SIZE, X_tfidf_train.shape[0])
rng = np.random.default_rng(RANDOM_STATE)
point_idx = np.sort(rng.choice(X_tfidf_train.shape[0], size=n_points, replace=False))

svd = TruncatedSVD(n_components=2, random_state=RANDOM_STATE)
coords = svd.fit_transform(X_tfidf_train[point_idx])
var = svd.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(9, 6.5))
# label kecil digambar paling atas (zorder lebih besar) agar tidak tertutup Bug Aplikasi
size_rank = {lab: rank for rank, lab in enumerate(label_counts.sort_values(ascending=False).index)}
for lab in LABELS:
    mask = y_train[point_idx] == lab
    ax.scatter(coords[mask, 0], coords[mask, 1], s=10, alpha=0.55, color=LABEL_COLORS[lab],
               label=f'{lab} ({mask.sum():,})', edgecolors='white', linewidths=0.3, zorder=2 + size_rank[lab])
ax.set_xlabel(f'Komponen 1 ({var[0]:.1f} persen variansi)')
ax.set_ylabel(f'Komponen 2 ({var[1]:.1f} persen variansi)')
ax.set_title(f'Proyeksi TruncatedSVD 2D dari TF-IDF kata ({n_points:,} ulasan latih)')
ax.legend(title='Label', markerscale=2, frameon=False, loc='best')
plt.tight_layout()
save_fig(fig, '03_tfidf_svd_2d')
plt.show()

## 11. Kelebihan dan kekurangan

| Aspek | Bag of Words | TF-IDF kata | TF-IDF char n-gram |
|---|---|---|---|
| Cara kerja | Hitungan kata | Hitungan kata dikali bobot kekhasan, dinormalisasi | Seperti TF-IDF kata, tetapi unitnya potongan 3 sampai 5 huruf |
| Kesederhanaan | Paling sederhana dan mudah dijelaskan | Sederhana, satu langkah tambahan (IDF) | Sederhana, tetapi fiturnya sulit dibaca manusia |
| Kata umum | Mendominasi vektor | Bobotnya diturunkan oleh IDF | Bobotnya diturunkan oleh IDF |
| Panjang ulasan | Ulasan panjang punya nilai lebih besar | Dinetralkan oleh normalisasi L2 | Dinetralkan oleh normalisasi L2 |
| Typo dan variasi ejaan | Dianggap kata berbeda, sering di luar kosakata | Sama dengan BoW | Tahan typo, karena ejaan mirip berbagi potongan huruf (lihat 7.1) |
| Imbuhan dan kata turunan | Tidak terkait | Tidak terkait | Sebagian terkait lewat potongan huruf yang sama |
| Urutan kata | Hanya lewat bigram | Hanya lewat bigram | Hilang antar kata (potongan tidak melintasi kata) |
| Makna dan sinonim | Tidak tertangkap (`lemot` dan `lambat` berbeda) | Tidak tertangkap | Tidak tertangkap |
| Dimensi dan biaya | Tinggi, sangat sparse | Sama dengan BoW | Paling tinggi, fitur aktif per ulasan paling banyak, paling lambat (lihat 8) |
| Kecocokan model | Cocok untuk Multinomial Naive Bayes (hitungan) | Cocok untuk SVM dan Logistic Regression | Cocok untuk SVM dan Logistic Regression |

Keterbatasan dalam menangkap makna dan sinonim menjadi motivasi notebook 04 (word embedding Word2Vec dan FastText).

## 12. Menyimpan artefak untuk notebook 03

**Pilihan artefak.** Notebook ini menyimpan **split dan konfigurasi**, bukan matriks atau vectorizer hasil pickle. Alasannya:

1. Notebook 03 memakai `Pipeline` scikit-learn (vectorizer + model). Vectorizer di-fit ulang di dalam pipeline hanya pada data latih, sehingga hasilnya sama persis dengan notebook ini, dan pipeline yang sama dapat langsung disimpan untuk aplikasi Part 2.
2. Fitting ketiga vectorizer hanya butuh beberapa detik, jadi tidak ada penghematan berarti dari menyimpan matriks.
3. File pickle bergantung pada versi scikit-learn, sedangkan CSV dan JSON dapat dibaca di mana saja dan mudah diperiksa.

File yang disimpan di `/kaggle/working`:

| File | Isi |
|---|---|
| `dataset_split.csv` | Seluruh kolom `clean_dataset_part1.csv` ditambah `row_id` dan `split` (`train` atau `test`). Ini satu-satunya Input yang dibutuhkan notebook 03 |
| `representation_config.json` | Parameter ketiga vectorizer, `random_state`, `test_size`, kolom teks, dan urutan `LABELS` |
| `feature_summary.csv` | Tabel perbandingan bagian 8 |
| `top_terms_per_label.csv` | Term teratas per label (rata-rata TF-IDF dan chi-square) |
| `sparse_examples.csv` | Contoh nilai bukan nol untuk ulasan contoh pada ketiga representasi |
| `typo_similarity.csv` | Tabel ketahanan typo bagian 7.1 |
| `figures/*.png` | Grafik untuk laporan dan slide (200 dpi) |

Cara notebook 03 memakai artefak ini:

```python
df = pd.read_csv(split_path, keep_default_na=False)
config = json.load(open(config_path))
train_df, test_df = df[df['split'] == 'train'], df[df['split'] == 'test']
params = dict(config['representations']['tfidf_word']['params'])
params['ngram_range'] = tuple(params['ngram_range'])
pipeline = Pipeline([('vectorizer', TfidfVectorizer(**params)), ('model', LinearSVC(class_weight='balanced'))])
```

In [ ]:
assert df['split'].isin(['train', 'test']).all()
assert (df['split'] == 'test').sum() == len(test_ids)

df.to_csv(f'{OUTPUT_DIR}/{SPLIT_FILE}', index=False)

config = {
    'source_file': CLEAN_FILE,
    'split_file': SPLIT_FILE,
    'text_column': TEXT_COLUMN,
    'label_column': 'label',
    'labels': LABELS,
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'n_train': int((df['split'] == 'train').sum()),
    'n_test': int((df['split'] == 'test').sum()),
    'run_type': RUN_TYPE,
    'representations': REPRESENTATIONS,
}
with open(f'{OUTPUT_DIR}/{CONFIG_FILE}', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

feature_summary.to_csv(f'{OUTPUT_DIR}/feature_summary.csv')
top_terms.to_csv(f'{OUTPUT_DIR}/top_terms_per_label.csv', index=False)
pd.concat({key: sparse_triplets(key, n_rows=40) for key in REPRESENTATIONS}, names=['representasi']) \
    .reset_index(level=0).to_csv(f'{OUTPUT_DIR}/sparse_examples.csv', index=False)
typo_table.to_csv(f'{OUTPUT_DIR}/typo_similarity.csv', index=False)

# pemeriksaan: file split dapat dibaca ulang dan sama dengan data di memori
check = pd.read_csv(f'{OUTPUT_DIR}/{SPLIT_FILE}', keep_default_na=False)
assert len(check) == len(df) and (check['split'].to_numpy() == df['split'].to_numpy()).all()

print(f'Tersimpan: {SPLIT_FILE} ({len(df):,} baris; latih {config["n_train"]:,}, uji {config["n_test"]:,})')
print('Isi OUTPUT_DIR:', sorted(os.listdir(OUTPUT_DIR)))

## 13. Penutup

Ringkasan notebook ini:
1. `clean_dataset_part1.csv` dibagi 80 persen latih dan 20 persen uji secara stratified dengan `random_state=42`. Split disimpan di `dataset_split.csv` agar notebook 03 dan model Part 2 memakai data uji yang sama.
2. Tiga representasi dibangun dari data latih: Bag of Words dan TF-IDF kata (unigram dan bigram, `min_df=2`) dengan kosakata yang sama, serta TF-IDF char n-gram (`char_wb`, 3 sampai 5 huruf, `min_df=5`).
3. Jumlah fitur, sparsity, contoh matriks sparse, nilai IDF, term terpenting per label, dan ketahanan terhadap typo ditampilkan sebagai bahan laporan.

Catatan untuk notebook 03:
- Latih NB, SVM, LR, dan RF pada ketiga representasi, dengan `class_weight='balanced'` (kecuali NB) dan F1 macro sebagai metrik utama.
- Bandingkan TF-IDF kata dan char n-gram, terutama pada label kecil (Desain Antarmuka, Permintaan Fitur).
- Random Forest pada char n-gram akan paling lambat karena jumlah fiturnya besar.

**Notebook berikutnya:** `03_classical_ml.ipynb` (nama Kaggle `03-ml-classification-app-feedback`), dengan Input Output notebook ini (`dataset_split.csv` dan `representation_config.json`).